# Survival Analysis and Conversion Forecasting

This notebook covers time-to-event modeling: Kaplan-Meier estimation, Cox Proportional Hazards, Random Survival Forests, and evaluation with censoring-aware metrics (C-index, Integrated Brier Score).

## Simulating Marketing Conversion Data

We generate right-censored marketing data where users from different acquisition channels convert at different rates. Users who haven't converted by the observation window end are censored.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 2000

channel = np.random.choice(["Search", "Social"], size=n_samples)
ad_spend = np.random.gamma(2, 40, n_samples)
page_views = np.random.poisson(4, n_samples)

# Search converts faster than Social; ad_spend and page_views also influence
base_hazard = np.where(channel == "Search", 0.06, 0.03)
hazard = base_hazard + 0.0003 * ad_spend + 0.002 * page_views
time_to_event = np.random.exponential(1 / hazard)

observation_window = 30
observed = time_to_event <= observation_window
duration = np.minimum(time_to_event, observation_window)

df = pd.DataFrame({
    "duration": duration,
    "event": observed.astype(int),
    "channel": channel,
    "ad_spend": ad_spend,
    "page_views": page_views,
})

print(f"Dataset: {len(df)} users")
print(f"Converted: {df['event'].sum()} ({df['event'].mean():.1%})")
print(f"Censored: {(1 - df['event']).sum()} ({1 - df['event'].mean():.1%})")

## Kaplan-Meier Survival Curves

The Kaplan-Meier estimator provides a non-parametric baseline for the survival function by channel.

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter()

fig, ax = plt.subplots(figsize=(10, 6))
for ch in df["channel"].unique():
    mask = df["channel"] == ch
    kmf.fit(df[mask]["duration"], event_observed=df[mask]["event"], label=ch)
    kmf.plot_survival_function(ax=ax)

ax.set_title("Time to Conversion by Channel")
ax.set_xlabel("Days Since Acquisition")
ax.set_ylabel("Probability of Non-Conversion")
plt.tight_layout()
plt.show()

## Cox Proportional Hazards Model

The Cox PH model quantifies how covariates multiplicatively affect the hazard rate. Hazard ratios < 1 indicate risk reduction.

In [ ]:
from lifelines import CoxPHFitter

df_cox = df.copy()
df_cox["channel_search"] = (df_cox["channel"] == "Search").astype(int)
df_cox = df_cox.drop(columns=["channel"])

cph = CoxPHFitter(penalizer=0.05)
cph.fit(
    df_cox,
    duration_col="duration",
    event_col="event",
    robust=True,
)

cph.print_summary(decimals=3)

In [ ]:
# Visualize hazard ratios
cph.plot()
plt.title("Cox PH Hazard Ratios")
plt.tight_layout()
plt.show()

## Random Survival Forest

Tree-based survival models capture non-linear interactions and output individualized survival functions for multi-horizon probabilistic forecasting.

In [ ]:
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.model_selection import train_test_split

# Prepare structured survival array
y_surv = Surv.from_arrays(event=df["event"].astype(bool), time=df["duration"])
X = pd.get_dummies(df[["ad_spend", "page_views", "channel"]], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_surv, test_size=0.3, random_state=42
)

rsf = RandomSurvivalForest(
    n_estimators=200,
    min_samples_leaf=15,
    n_jobs=-1,
    random_state=42,
)

rsf.fit(X_train, y_train)
print("Random Survival Forest fitted.")

## Multi-Horizon Probabilistic Forecasts

By evaluating 1 - S(t) at specific horizons, we get individualized conversion probability forecasts.

In [ ]:
horizons = [7, 14, 30]

surv_funcs = rsf.predict_survival_function(X_test)

forecasts = []
for fn in surv_funcs:
    prob_converted = [1 - fn(t) for t in horizons]
    forecasts.append(prob_converted)

forecast_df = pd.DataFrame(
    forecasts,
    columns=[f"prob_convert_{h}d" for h in horizons],
    index=X_test.index,
)

print("Multi-horizon conversion forecasts:")
forecast_df.head(10)

In [ ]:
# Plot individual survival curves for a few test subjects
fig, ax = plt.subplots(figsize=(10, 6))

for i, fn in enumerate(surv_funcs[:5]):
    ax.step(fn.x, fn(fn.x), where="post", label=f"User {X_test.index[i]}")

ax.set_title("Individual Survival Curves (Random Survival Forest)")
ax.set_xlabel("Days")
ax.set_ylabel("Survival Probability")
ax.legend()
plt.tight_layout()
plt.show()

## Evaluating Time-to-Event Calibration

We evaluate with:
- **Harrell's C-index**: discrimination (ability to rank subjects by risk)
- **Integrated Brier Score**: calibration (accuracy of predicted probabilities)

In [ ]:
from sksurv.metrics import concordance_index_censored, integrated_brier_score

# Discrimination: C-index
risk_scores = rsf.predict(X_test)
c_index, _, _, _, _ = concordance_index_censored(
    y_test["event"], y_test["time"], risk_scores
)
print(f"C-index: {c_index:.4f}")

# Calibration: Integrated Brier Score
times = np.percentile(y_train["time"], [25, 50, 75])
preds = np.row_stack([fn(times) for fn in surv_funcs])
ibs = integrated_brier_score(y_train, y_test, preds, times)
print(f"Integrated Brier Score: {ibs:.4f}")